# Hash table


## Reading

- [Hashing](https://opendsa-server.cs.vt.edu/ODSA/Books/Everything/html/index.html#hashing) and probing techniques from the OpenDSA project.


## Brief history

The idea of using a mathematical function to map data to storage locations dates back to the early 1950s. Hans Peter Luhn, an IBM researcher, is often credited with proposing the concept around 1953. The term _hash_ likely comes from the idea of "chopping up" or scrambling data to produce an address. By the late 1960s and 1970s, hash tables had become a standard topic in computer science textbooks and a go-to data structure for fast lookups. Today, hash tables power Python's `dict`, Java's `HashMap`, and similar constructs in other languages.


## Why hash tables matter

Most data structures force a trade-off between speed and simplicity. A plain list lets you store anything, but finding a specific item takes $\mathcal{O}(n)$ time because you may have to scan every element. A sorted list improves search to $\mathcal{O}(\log n)$ with binary search, but inserting a new item is expensive because you have to keep things in order.

Hash tables offer something better: on average, both **insertion** and **lookup** run in $\mathcal{O}(1)$ time. That constant-time performance is what makes hash tables the backbone of dictionaries, caches, symbol tables in compilers, database indexes, and countless other systems.

## Keys and values

Typically a hash table stores **(key, value)** pairs, where each entry is identified by a unique **key**.

Think of it like a contacts list on your phone. The **key** is the person's name (it's what you search by) and the **value** is their phone number (it's the data you actually want). For example:

| Key (name) | Value (phone number) |
| ---------- | -------------------- |
| `"Alice"`  | `"312-555-0101"`     |
| `"Bob"`    | `"773-555-0202"`     |
| `"Carol"`  | `"847-555-0303"`     |

You look up `"Alice"` (the key) and get back `"312-555-0101"` (the value). The hash table uses a hash function on the key to decide where in its underlying array to store and later find that pair.

**Simplification in our code:** In the two classes below, we store just string values (people's names). We use `abs(hash(name))` to compute a numeric "key" which is then mapped to a position in the underlying array via the modulo operator (`%`). This keeps the examples focused on the mechanics of hashing and collisions rather than on key-value bookkeeping.

## How a hash table works (in brief)

1. **Hash the key.** Pass the key through a hash function to get an integer.
2. **Map to a position.** Use `hash_value % array_size` to find an index in the underlying array.
3. **Handle collisions.** When two keys map to the same index, a collision resolution strategy kicks in. Our classes use **separate chaining**: each array slot holds a linked list, and colliding items are appended to that list.

The **load factor** is the ratio of occupied slots to total slots. When it exceeds a threshold (0.75 in our code), the table needs to grow.

## The `Node` class

Both hash table implementations below use separate chaining with a simple singly-linked list. Each node stores a name and a pointer to the next node.


In [41]:
from __future__ import annotations


class Node:
    """ "Simple linked list node"""

    def __init__(self, name: str) -> None:
        self._name: str = name
        self._next: None | Node = None

    def get_next(self):
        return self._next

    def set_next(self, node):
        self._next = node

    def get_name(self):
        return self._name

## `SimpleHashTable` — resize without rehashing

`SimpleHashTable` demonstrates the basic mechanics of a hash table: hashing, modular indexing, separate chaining, load-factor monitoring, and resizing. When the load factor hits the threshold, it calls `_resize()`.


In [ ]:
import copy


class SimpleHashTable:

    _DEFAULT_SIZE: int = 4
    _DEFAULT_THRESHOLD: float = 0.75
    _DEFAULT_RESIZE_BY: int = 2

    def __init__(self, size: int = _DEFAULT_SIZE) -> None:
        self._size: int = size
        self._underlying: list[None | Node] = [None] * self._size
        # How many element of the underlying array are not Nond
        self._occupied: int = 0
        # How many nodes are in the linked lists of the underlying array
        self._count: int = 0

    def add(self, name: str) -> None:
        """Assume that name is not empty and it is a legit person's name"""
        if self._load_factor() >= self._DEFAULT_THRESHOLD:
            # The load factor is greater than or equal to the threshold,
            # so it's time toresize the underlying array
            self._resize()
        # Compute the hash of the name and make it non-negative
        h: int = abs(hash(name))
        # Where in the underlying array the node with this name will go to
        position: int = h % self._size
        # Create a node with the name
        new_node = Node(name)
        # Determine if the position in the underlying array is occupied.
        # If it is not, then we can just put the new node there. If it is
        # occupied, then we need to add the new node to the linked list at
        # that position. We can do that by setting the next of the new node
        # to the current head of the linked list at that position and then
        # setting the head of the linked list at that position to the new node.
        if self._underlying[position] is None:
            self._occupied += 1
        else:
            new_node.set_next(self._underlying[position])
        # Finally, we can put the new node at the position in the underlying array
        self._underlying[position] = new_node
        # Increment the count of nodes in the linked lists of the underlying array
        self._count += 1

    def _resize(self):
        """Resize the underlying array by creating a new array that is twice as big as the current one and copying the elements from the current array to the new array. Then replace the current array with the new array."""
        # We want to replace the underlying array with a new array that is twice as big
        # as the current one. We can do that by creating a new array of the appropriate
        # size and then copying the elements from the current array to the new array.
        # Finally, we can replace the current array with the new array.
        self._size = self._DEFAULT_RESIZE_BY * len(self._underlying)
        temp: list[None | Node] = [None] * (self._size)
        for i in range(len(self._underlying)):
            temp[i] = self._underlying[i]
        # Replace underlying with temp
        self._underlying = copy.deepcopy(temp)

    def _load_factor(self) -> float:
        """Return the load factor of the hash table as the number of occupied
        elements divided by the size of the underlying array."""
        return self._occupied / self._size

    # Format constants for the __str__ method
    _FMT_HEADER: str = "There are {} element(s) with {} nodes"
    _FMT_DESCRIPTION: str = "The load factor is {}/{} ({:.2f})"
    _FMT_ELEMENT: str = "[{:<3}]: "
    _FMT_LINK: str = " --> {}"

    def __str__(self) -> str:
        """Return a string representation of the hash table."""
        strings = [self._FMT_HEADER.format(self._size, self._count)]
        strings.append(
            self._FMT_DESCRIPTION.format(
                self._occupied, self._size, self._load_factor()
            )
        )
        for i in range(self._size):
            element = self._FMT_ELEMENT.format(i)
            head = self._underlying[i]
            while head is not None:
                element += self._FMT_LINK.format(head.get_name())
                head = head.get_next()
            strings.append(element)
        return "\n".join(strings)

### The problem with `_resize()`

Look carefully at `_resize()` above. It doubles the array and copies each linked list into the **same index** it occupied before. It does **not** recompute `hash(name) % new_size` for any element.

Why is this a problem? Because `position = hash(name) % size`. When `size` changes, the correct position for each element may change too. By simply copying, elements can end up at indices that no longer match their hash, which means future lookups could fail or degrade.

This is intentional — `SimpleHashTable` is a teaching tool that shows what happens when you skip the rehashing step.

## `HashTable` — resize _with_ rehashing

`HashTable` fixes the problem above. When the load factor is exceeded, it calls `_rehash()` instead of `_resize()`.


In [43]:
from __future__ import annotations


class HashTable:
    """A hash table that uses separate chaining to resolve collisions.
    This is build as a more robust object to demonstrate the resizing
    of the underlying array together with rehashing of its elements at
    the same time. In the object SimpleHashTable above, after we resized
    the underlying array, we did not rehash its elements. Instead we
    just copied them to the new array. That is not how a real hash table
    works. In a real hash table, after we resize the underlying array, we
    need to rehash its elements and put them in the appropriate positions in
    the new array. This is because the hash of an element is computed based
    on the size of the underlying array. So when we resize the underlying
    array, the hash of each element will change and we need to put it in the
    appropriate position in the new array based on its new hash.
    """

    _DEFAULT_SIZE: int = 4
    _DEFAULT_THRESHOLD: float = 0.75
    _DEFAULT_RESIZE_BY: int = 2

    def __init__(self, size: int = 4) -> None:
        self._size: int = size
        self._underlying: list[None | Node] = [None] * self._size
        # How many element of the underlying array are not Nond
        self._occupied: int = 0
        # How many nodes are in the linked lists of the underlying array
        self._count: int = 0

    def _load_factor(self) -> float:
        """Return the load factor of the hash table as the number of occupied
        elements divided by the size of the underlying array."""
        return self._occupied / self._size

    def get_size(self) -> int:
        """Return the size of the underlying array."""
        return self._size

    def add(self, name: str) -> None:
        """Assume that name is not empty and it is a legit person's name"""
        if self._load_factor() >= self._DEFAULT_THRESHOLD:
            # The load factor is greater than or equal to the threshold,
            # so it's time to resize the underlying array
            self._rehash()
        # Compute the hash of the name and make it non-negative
        h: int = abs(hash(name))
        # Where in the underlying array the node with this name will go to
        position: int = h % self._size
        # Create a node with the name
        new_node = Node(name)
        # Determine if the position in the underlying array is occupied.
        # If it is not, then we can just put the new node there. If it is
        # occupied, then we need to add the new node to the linked list at
        # that position. We can do that by setting the next of the new node
        # to the current head of the linked list at that position and then
        # setting the head of the linked list at that position to the new node.
        if self._underlying[position] is None:
            self._occupied += 1
        else:
            new_node.set_next(self._underlying[position])
        # Finally, we can put the new node at the position in the underlying array
        self._underlying[position] = new_node
        # Increment the count of nodes in the linked lists of the underlying array
        self._count += 1

    def _rehash(self):
        """Resize the underlying array by creating a new HashTable object with the appropriate size and then adding all the elements from the current hash table
        to the new hash table. Finally, we can replace the current hash table with the
        new hash table."""
        # Create a new hash larger than the current one by a factor of _DEFAULT_RESIZE_BY
        temp: HashTable = HashTable(self._DEFAULT_RESIZE_BY * self.get_size())
        # Add all the elements from the current hash table to the new hash table by
        # iterating over all the nodes in the current hash table and adding their
        # names to the new hash table.
        for node in self:
            temp.add(node.get_name())
        # Replace underlying fields with fields of the temporary hash table, by
        # calling the _get_status method of the temporary object to get its size,
        # underlying array, number of occupied elements, and number of nodes in
        # the linked lists and then assigning those values to the corresponding
        # fields of the current object.
        self._size, self._underlying, self._occupied, self._count = temp._get_status()

    def __iter__(self):
        """Iterator for all the linked nodes present in the hash table.
        Makes the object iterable with a for loop to iterate over all
        the nodes in the hash table."""
        for i in range(self._size):
            current = self._underlying[i]
            while current is not None:
                yield current
                current = current.get_next()

    def _get_status(self):
        """Return the status of the hash table as a tuple of the size of the underlying array, the underlying array itself, the number of occupied elements in the underlying array, and the number of nodes in the linked lists of the underlying array."""
        return self._size, self._underlying, self._occupied, self._count

    # Format constants for the __str__ method
    _FMT_HEADER: str = "There are {} element(s) with {} nodes"
    _FMT_DESCRIPTION: str = "The load factor is {}/{} ({:.2f})"
    _FMT_ELEMENT: str = "[{:<3}]: "
    _FMT_LINK: str = " --> {}"

    def __str__(self) -> str:
        """Return a string representation of the hash table."""
        strings = [self._FMT_HEADER.format(self._size, self._count)]
        strings.append(
            self._FMT_DESCRIPTION.format(
                self._occupied, self._size, self._load_factor()
            )
        )
        for i in range(self._size):
            element = self._FMT_ELEMENT.format(i)
            head = self._underlying[i]
            while head is not None:
                element += self._FMT_LINK.format(head.get_name())
                head = head.get_next()
            strings.append(element)
        return "\n".join(strings)

### Why `_rehash()` is the correct approach

The `_rehash()` method creates a brand-new `HashTable` object with double the capacity and then **re-adds every element** from the old table. Because `add()` computes `hash(name) % new_size`, every element lands in the position that the new, larger array requires. Once all elements have been transferred, the method copies the new table's internal state back into `self` using method `_get_state()`.

Notice how this design is made cleaner by two features:

1. **`__iter__`**: The `HashTable` class implements `__iter__`, making the object iterable. This lets `_rehash()` simply write `for node in self:` to walk through every node across all chains — no manual index-and-pointer loop needed.

2. **Self-delegation**: `_rehash()` creates a new `HashTable` instance and uses its own `add()` method to insert elements. This avoids duplicating any hashing or collision logic. The new object handles everything correctly because it already knows how.

Compare this to `SimpleHashTable._resize()`, which manually allocates an array and copies slots by index. That approach is fragile and incorrect because it skips the essential re-positioning step.

## Side-by-side comparison

| Aspect                   | `SimpleHashTable`                         | `HashTable`                                   |
| ------------------------ | ----------------------------------------- | --------------------------------------------- |
| Resize method            | `_resize()`                               | `_rehash()`                                   |
| Creates new array?       | Yes                                       | Yes (via a new `HashTable` object)            |
| Recomputes positions?    | **No** — copies chains to same indices    | **Yes** — re-adds each element with `add()`   |
| Iterable?                | No                                        | Yes (`__iter__` with `yield`)                 |
| Correctness after resize | Broken — elements may be at wrong indices | Correct — every element is in its proper slot |
| Purpose                  | Teaching tool: shows what _not_ to do     | Demonstrates the real-world approach          |

## Key takeaways

1. A hash table maps keys to positions using a hash function and the modulo operator.
2. Collisions are resolved here with **separate chaining** (linked lists at each slot).
3. When the **load factor** exceeds a threshold, the table must grow.
4. After growing, every element must be **rehashed** against the new size — simply copying is not enough.
5. Average-case insertion and lookup are $\mathcal{O}(1)$; worst case (all keys collide) is $\mathcal{O}(n)$.
